# Heading levels from PDFs: Section Anything, Docling and pymupdf4llm on three papers

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Omniwot/section-anything/blob/main/notebooks/heading-levels.ipynb)

**What this shows.** Users of several PDF converters ask for heading *levels* instead of a flat list of
headings, so that a RAG pipeline can chunk by section and keep the heading path (open issues, by number:
docling #287, #1023, #2335, #2121; markitdown #304; unstructured #4204; marker #912). This notebook runs
three tools on three public papers and prints each tool's heading tree side by side:

- **Section Anything, Free mode** (no model, no key, $0): `section_anything.section(pdf, mode="free")`.
- **Docling, default pipeline**: every section header comes out at one level (`##`).
- **Docling with its opt-in heading hierarchy** (`HeadingHierarchyOptions(enabled=True)`, CLI `--heading-hierarchy`;
  added in 2026): levels from bookmarks, numbering and font style.
- **pymupdf4llm** (`to_markdown`, OCR off): header levels from its layout analysis and font sizes, no LLM.

**What it does not claim.** Three documents are a demonstration, not a benchmark. The large-sample numbers
are in the README (905 held-out READoc arXiv papers, heading-tree TEDS: Free 60.0, Docling 2.140 with
heading hierarchy on 49.4; research finding F001). Docling and pymupdf4llm convert whole documents
(tables, figures, formulas, OCR); only the heading tree is compared here, and Docling runs without OCR
and without table structure to keep memory low, which does not change its headings. No tool is "wrong"
below: each has a different idea of what a heading is, and the printed trees let you judge.

**Versions used when the saved outputs were made:** section-anything 0.1.9,
docling 2.150.0, pymupdf4llm 2.1, Python 3.11.18, CPU only, macOS.
Docling outputs in this notebook were produced in a subprocess (so its model stays out of the kernel's
memory); set `DOCLING_GUARD` to a wrapper command if you want to run it under a memory guard.

In [1]:
%pip install -q "section-anything==0.1.9" "docling==2.150.0" "pymupdf4llm==2.1"

Note: you may need to restart the kernel to use updated packages.


## 1. Download the papers

In [2]:
# Three public papers, shortest first. All CC BY 4.0; downloaded from arXiv when you run the notebook.
PAPERS = [
    ("2209.07299v2", "Knowledge Is Flat (Chen et al.), 13 pages, no PDF bookmarks; in READoc"),
    ("2304.08485v2", "Visual Instruction Tuning (Liu et al.), 25 pages, PDF bookmarks; the README's demo paper"),
    ("2305.06327v1", "Thermalization, Chaos and Hydrodynamics (Ganapa), 88 pages, PDF bookmarks, nested; in READoc"),
]

import os, time, urllib.request
from pathlib import Path

PDF_DIR = Path("pdfs"); PDF_DIR.mkdir(exist_ok=True)
for arxiv_id, _ in PAPERS:
    path = PDF_DIR / f"{arxiv_id}.pdf"
    if path.exists():
        continue
    req = urllib.request.Request(f"https://arxiv.org/pdf/{arxiv_id}", headers={"User-Agent": "section-anything-notebook"})
    with urllib.request.urlopen(req, timeout=120) as r:
        data = r.read()
    assert data[:5] == b"%PDF-", arxiv_id
    path.write_bytes(data)
    time.sleep(3)  # be gentle with arXiv
print(sorted(p.name for p in PDF_DIR.glob("*.pdf")))

['2209.07299v2.pdf', '2304.08485v2.pdf', '2305.06327v1.pdf']


## 2. One function per tool

Each returns rows of (Markdown depth, heading text, page) in reading order.

In [3]:
import json, os, shlex, subprocess, sys, time
import section_anything, pymupdf4llm

SECONDS = {}  # (pdf, tool) -> wall-clock seconds, CPU only

# Every tool's headings become rows (depth, text, page), where depth is the Markdown heading depth
# each tool itself would print: 1 for "#", 2 for "##", and so on.

def sa_rows(pdf):
    t = time.time()
    tree = section_anything.section(pdf, mode="free")
    SECONDS[(pdf, "Section Anything free")] = time.time() - t
    rows = []
    for s in tree.sections:
        depth = 1 if "doc_title" in s["flags"] else min(6, int(s["level"] or 1) + 1)  # same rule as tree.to_markdown()
        rows.append((depth, f"{s['number']} {s['title']}".strip(), f"p{s['page_start']}-{s['page_end']}"))
    return rows, tree

DOCLING_SCRIPT = r"""
import json, sys
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import HeadingHierarchyOptions, PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling_core.types.doc import SectionHeaderItem, TitleItem
pdf, hierarchy, out = sys.argv[1], sys.argv[2] == "1", sys.argv[3]
opts = PdfPipelineOptions(do_ocr=False, do_table_structure=False)  # headings only; keeps memory low
if hierarchy:
    opts.heading_hierarchy_options = HeadingHierarchyOptions(enabled=True)
doc = DocumentConverter(format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=opts)}).convert(pdf).document
rows = []
for item, _ in doc.iterate_items():
    page = item.prov[0].page_no if item.prov else None
    if isinstance(item, TitleItem):
        rows.append((1, item.text, f"p{page}"))
    elif isinstance(item, SectionHeaderItem):
        rows.append((min(6, item.level + 1), item.text, f"p{page}"))  # export_to_markdown prints level+1 hashes
json.dump(rows, open(out, "w"))
"""

def docling_rows(pdf, hierarchy):
    """Run Docling in a subprocess (optionally under $DOCLING_GUARD) and cache the headings next to the PDF."""
    out = f"{pdf}.docling-{'hier' if hierarchy else 'default'}.json"
    if not os.path.exists(out):
        guard = shlex.split(os.environ.get("DOCLING_GUARD", ""))
        t = time.time()
        r = subprocess.run([*guard, sys.executable, "-c", DOCLING_SCRIPT, pdf, "1" if hierarchy else "0", out],
                           capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(f"docling exited {r.returncode}: {r.stdout[-300:]} {r.stderr[-300:]}")
        json.dump({"rows": json.load(open(out)), "seconds": time.time() - t}, open(out, "w"))
    cached = json.load(open(out))
    SECONDS[(pdf, f"Docling {'heading hierarchy' if hierarchy else 'default'}")] = cached["seconds"]
    return [tuple(x) for x in cached["rows"]]

def pymupdf4llm_rows(pdf):
    rows = []
    t = time.time()
    chunks = pymupdf4llm.to_markdown(pdf, page_chunks=True, show_progress=False, use_ocr=False)  # text layer only, like the others
    SECONDS[(pdf, "pymupdf4llm")] = time.time() - t
    for chunk in chunks:
        page = chunk["metadata"]["page_number"]
        for line in chunk["text"].splitlines():
            if line.startswith("#"):
                hashes = len(line) - len(line.lstrip("#"))
                rows.append((hashes, line.lstrip("#").strip().strip("*_ "), f"p{page}"))
    return rows

def show(name, rows, limit=22):
    print(f"--- {name}: {len(rows)} headings, depths {sorted({r[0] for r in rows})}")
    for depth, text, page in rows[:limit]:
        print(f"{'  ' * (depth - 1)}{'#' * depth} {text[:60]}  [{page}]")
    if len(rows) > limit:
        print(f"  ... {len(rows) - limit} more")

def stats(rows):
    depths = sorted({r[0] for r in rows})
    return {"headings": len(rows), "distinct levels": len(depths), "levels present": ",".join(map(str, depths))}

## 3. The trees, side by side

The first 22 headings of each tree are printed; the table in the next cell counts all of them.

In [4]:
results = {}
for arxiv_id, about in PAPERS:
    pdf = str(PDF_DIR / f"{arxiv_id}.pdf")
    print("=" * 100); print(f"{arxiv_id}: {about}"); print("=" * 100)
    rows = {}
    rows["Section Anything free"], tree = sa_rows(pdf)
    rows["Docling default"] = docling_rows(pdf, hierarchy=False)
    rows["Docling heading hierarchy"] = docling_rows(pdf, hierarchy=True)
    rows["pymupdf4llm"] = pymupdf4llm_rows(pdf)
    for name, r in rows.items():
        show(name, r)
    results[arxiv_id] = rows
    print()

2209.07299v2: Knowledge Is Flat (Chen et al.), 13 pages, no PDF bookmarks; in READoc


--- Section Anything free: 26 headings, depths [1, 2, 3]
# Knowledge Is Flat: A Seq2Seq Generative Framework for Variou  [p1-1]
  ## Chen Chen1,∗, Yufei Wang2,∗, Bing Li3 and Kwok-Yan Lam1†  [p1-1]
    ### Nanyang Technological University, Singapore1 Macquarie Unive  [p1-1]
  ## Abstract  [p1-1]
  ## 1 Introduction  [p1-2]
    ### FB15K-237 and FB15K-237N for SKGC,  [p2-2]
  ## 3 Proposed Method  [p2-2]
  ## 2 Related work  [p2-3]
    ### 3.1 Knowledge Graph Completion  [p3-3]
    ### 3.2 A Seq2Seq Framework for KGC  [p3-3]
    ### 3.3 Entity & Relation Representation  [p3-4]
    ### Seq2Seq Dropout  [p4-4]
    ### 3.4 KGC Inference  [p4-5]
  ## 4 Experiment  [p5-5]
    ### 4.2 Experimental results  [p5-5]
    ### 4.1 Experimental Settings  [p5-7]
    ### 4.3 Ablation Study  [p7-7]
    ### Impact of Decoding Methods  [p8-8]
    ### 4.4 Discussion  [p8-9]
    ### Parameter Growth Since KG-S2S represents  [p9-9]
  ## 5 Conclusion and Future Work  [p9-9]
  ## Acknowledgement  [p9-10]
  ..

--- Section Anything free: 19 headings, depths [1, 2, 3]
# Visual Instruction Tuning  [p1-1]
  ## Abstract  [p1-1]
  ## 1 Introduction  [p1-2]
  ## 2 Related Work  [p2-2]
  ## 3 GPT-assisted Visual Instruction Data Generation  [p2-4]
  ## 4 Visual Instruction Tuning  [p4-4]
    ### 4.1 Architecture  [p4-4]
    ### 4.2 Training  [p4-5]
  ## 5 Experiments  [p5-5]
    ### 5.1 Multimodal Chatbot  [p5-8]
    ### 5.2 ScienceQA  [p8-9]
  ## 6 Conclusion  [p9-10]
  ## References  [p10-13]
  ## A Broader Impact  [p14-14]
  ## B More Results  [p14-19]
  ## C Training Details  [p20-20]
  ## D Assets  [p20-20]
  ## E Data  [p20-21]
  ## F Prompts  [p21-25]
--- Docling default: 30 headings, depths [2]
  ## Visual Instruction Tuning  [p1]
  ## Abstract  [p1]
  ## 1 Introduction  [p1]
  ## 2 Related Work  [p2]
  ## 3 GPT-assisted Visual Instruction Data Generation  [p2]
  ## 4.2 Training  [p4]
  ## 5 Experiments  [p5]
  ## 5.1 Multimodal Chatbot  [p5]
  ## Visual input example, Extreme Ironing:  [p6]

--- Section Anything free: 61 headings, depths [3, 4]
    ### Front matter  [p1-4]
    ### Acknowledgements  [p5-10]
    ### Chapter 1 Introduction  [p11-11]
    ### 1.1 A General Introduction to this Work  [p12-12]
      #### 1.1.1 Thermalization and its Importance in Statistical Physi  [p12-12]
      #### 1.1.2 Evolution Before Thermalization - Description in Terms  [p13-13]
      #### 1.1.3 Structure of the Thesis  [p13-14]
    ### 1.2 Problem Statements  [p14-14]
      #### 1.2.1 Thermalization in the α−FPUT Chain  [p14-17]
      #### 1.2.2 Blast Wave in an Alternate Mass Hard Particle Gas  [p17-19]
    ### 1.3 Methods Used in this Study  [p19-19]
      #### 1.3.1 Symplectic Integrator  [p19-20]
      #### 1.3.2 Equipartition Theorem  [p20-20]
      #### 1.3.3 Averaging Methods  [p20-22]
      #### 1.3.4 Estimation of Equilibration Time  [p22-22]
      #### 1.3.5 Navier-Stokes-Fourier (NSF) Equations and the MacCor-   [p22-23]
    ### Chapter 2 The Fermi-Pasta-Ulam-Tsingou Story  [

## 4. Headings found, distinct levels, levels present

In [5]:
print(f"{'paper':<14}{'tool':<28}{'headings':>9}{'distinct levels':>17}  {'levels present':<16}{'seconds':>8}")
for arxiv_id, rows in results.items():
    pdf = str(PDF_DIR / f"{arxiv_id}.pdf")
    for name, r in rows.items():
        s = stats(r)
        print(f"{arxiv_id:<14}{name:<28}{s['headings']:>9}{s['distinct levels']:>17}  {s['levels present']:<16}{SECONDS[(pdf, name)]:>8.1f}")

paper         tool                         headings  distinct levels  levels present   seconds
2209.07299v2  Section Anything free              26                3  1,2,3                0.9
2209.07299v2  Docling default                    19                1  2                  375.3
2209.07299v2  Docling heading hierarchy          19                2  2,3                  9.1
2209.07299v2  pymupdf4llm                        19                3  1,2,3                2.6
2304.08485v2  Section Anything free              19                3  1,2,3                2.9
2304.08485v2  Docling default                    30                1  2                   12.1
2304.08485v2  Docling heading hierarchy          30                4  2,3,5,6             12.1
2304.08485v2  pymupdf4llm                        27                4  1,2,3,4              3.1
2305.06327v1  Section Anything free              61                2  3,4                  3.7
2305.06327v1  Docling default                    7

## Reading the trees

For reference, READoc's answer keys (not loaded here) list 19 headings over 4 levels for 2209.07299 and 69 headings
over 5 levels for 2305.06327; 2304.08485 is not in READoc. What the printed trees show, paper by paper:

- **2209.07299 (no bookmarks).** Docling and pymupdf4llm both find exactly the paper's 19 headings. Docling default
  prints them flat; with the hierarchy on, the numbered subsections nest correctly, but Abstract, Acknowledgement,
  References and the appendices are placed one level under the section before them. pymupdf4llm's tree is the
  cleanest of the four on this paper. Section Anything Free, with no outline to lean on, adds seven false headings
  (the author line, affiliations, bold run-in labels, two lines of body text) and its reading order on the
  two-column page puts "3 Proposed Method" before "2 Related work". This is Free's known weak spot: on READoc
  arXiv papers without an outline it scores below Docling (42.3 against 46.7, finding F001); the paid Quick and
  Standard modes are the fix for such documents.
- **2304.08485 (bookmarks).** Section Anything Free returns the paper's 19 real headings at the right levels, from
  the bookmarks. Docling misses "4 Visual Instruction Tuning" and "4.1 Architecture" in both settings and takes
  figure and prompt labels ("Visual input example, Extreme Ironing:", "Context type 1: Captions") as headers; with
  the hierarchy on those labels drop to `#####` and References nests under "6 Conclusion". pymupdf4llm gets the
  numbered tree right and puts the labels at `####`.
- **2305.06327 (88-page thesis, bookmarks).** Docling with the hierarchy on and pymupdf4llm both give four levels
  with chapter, section and subsection nesting; both split "Chapter 1" and "Introduction" into two headings and
  list the title-page items as headings. Section Anything Free merges "Chapter 1 Introduction", folds the title
  pages into front matter, but puts chapters and their sections at the same level and finds no document title,
  so its tree has only two levels here against the answer key's five.

In short: Docling's hierarchy option and pymupdf4llm both turn a flat list into a usable tree on these papers, and
each has its own failure modes (missed or extra headers, back matter nested under the last section, split chapter
titles). Section Anything Free is strongest when the PDF carries bookmarks or a printed contents page and weakest
without them. The large-sample picture (905 held-out READoc papers, heading-tree TEDS): Free 60.0, Docling 2.140
with hierarchy on 49.4, Quick 68.0, Standard 70.0 (README, finding F001); on GitHub READMEs Free ties with Docling.
Wall-clock seconds in the table are one laptop, CPU only; each Docling run is a fresh subprocess, so its seconds
include loading its models, and the first Docling run of the session (375 s here) also paid for a cold model load on
a machine under memory pressure; the hierarchy stage itself adds well under a second.

Three documents are not a benchmark. If your documents look different, run this notebook on them: every tool
here is free to run locally.